# Entrenamiento de Modelo de Detección de Vulnerabilidades en TypeScript/JavaScript

Este notebook entrena un clasificador de Machine Learning para detectar vulnerabilidades en código Node.js + Express + TypeScript.

**Objetivo**: Clasificar código como **SEGURO** o **VULNERABLE**

**Dataset**: Ejemplos sintéticos y reales de código vulnerable y seguro

**Características**:
- Tokens del código fuente
- Métricas del AST
- Llamadas a funciones peligrosas
- Presencia de sanitización y validación

**Modelos**: Random Forest, XGBoost, SVM, Logistic Regression

**Métrica mínima requerida**: F1-Score ≥ 0.85

In [ ]:
# Instalar dependencias (descomentar si es necesario)
# !pip install scikit-learn pandas numpy joblib xgboost esprima matplotlib seaborn

import warnings
warnings.filterwarnings('ignore')

# Librerías estándar
import os
import sys
import json
import re
from pathlib import Path
from typing import Dict, List, Tuple, Any

# Análisis de datos
import pandas as pd
import numpy as np

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
from xgboost import XGBClassifier
import joblib

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Bibliotecas importadas correctamente")
print(f"Versión de sklearn: {__import__('sklearn').__version__}")
print(f"Versión de pandas: {pd.__version__}")

In [ ]:
# Dataset sintético de código vulnerable y seguro
vulnerable_code_samples = [
    # NoSQL Injection
    """
    router.post('/login', async (req, res) => {
        const user = await User.findOne({ username: req.body.username, password: req.body.password });
        res.json(user);
    });
    """,
    """
    app.get('/search', async (req, res) => {
        const query = { name: req.query.name };
        const results = await Collection.find(query);
        res.json(results);
    });
    """,
    # Command Injection
    """
    const { exec } = require('child_process');
    router.post('/run', (req, res) => {
        exec('ls ' + req.body.directory, (error, stdout) => {
            res.send(stdout);
        });
    });
    """,
    """
    import { spawn } from 'child_process';
    app.post('/convert', (req, res) => {
        const file = req.query.file;
        const child = spawn('convert', [file]);
    });
    """,
    # Code Injection
    """
    router.post('/calculate', (req, res) => {
        const result = eval(req.body.expression);
        res.json({ result });
    });
    """,
    """
    app.get('/execute', (req, res) => {
        const func = new Function(req.query.code);
        func();
    });
    """,
    # Path Traversal
    """
    import fs from 'fs';
    router.get('/file', (req, res) => {
        const content = fs.readFileSync(req.params.filename);
        res.send(content);
    });
    """,
    """
    app.post('/upload', (req, res) => {
        fs.writeFileSync('./uploads/' + req.body.filename, req.body.data);
    });
    """,
    # Validación Insuficiente
    """
    router.post('/update', async (req, res) => {
        await User.updateOne({ _id: req.params.id }, req.body);
    });
    """,
    """
    app.delete('/remove', async (req, res) => {
        await Product.deleteMany({ category: req.query.category });
    });
    """,
]

secure_code_samples = [
    # Con validación
    """
    import { body, validationResult } from 'express-validator';
    import sanitize from 'express-mongo-sanitize';
    
    router.post('/login', [
        body('username').isAlphanumeric().trim(),
        body('password').isLength({ min: 8 })
    ], async (req, res) => {
        const errors = validationResult(req);
        if (!errors.isEmpty()) return res.status(400).json({ errors: errors.array() });
        const user = await User.findOne({ username: req.body.username });
        const isValid = await bcrypt.compare(req.body.password, user.password);
        if (isValid) res.json({ token: generateToken(user) });
    });
    """,
    # Con sanitización
    """
    import xss from 'xss-clean';
    import mongoSanitize from 'express-mongo-sanitize';
    
    app.use(xss());
    app.use(mongoSanitize());
    
    router.get('/search', [
        query('name').trim().escape()
    ], async (req, res) => {
        const results = await Collection.find({ name: req.query.name });
        res.json(results);
    });
    """,
    # Sin ejecución de comandos
    """
    import path from 'path';
    import { promisify } from 'util';
    
    router.get('/files', async (req, res) => {
        const files = await fs.readdir('./public');
        res.json(files);
    });
    """,
    # Con validación de rutas
    """
    import path from 'path';
    import fs from 'fs';
    
    router.get('/download', [
        param('filename').matches(/^[a-zA-Z0-9_.-]+$/)
    ], (req, res) => {
        const safePath = path.join(__dirname, 'files', path.basename(req.params.filename));
        if (!safePath.startsWith(__dirname)) return res.status(403).send('Forbidden');
        const content = fs.readFileSync(safePath);
        res.send(content);
    });
    """,
    # Con autenticación y autorización
    """
    import jwt from 'jsonwebtoken';
    import { authenticate, authorize } from './middleware';
    
    router.put('/users/:id', [
        authenticate,
        authorize(['admin']),
        body('email').isEmail(),
        body('name').trim().notEmpty()
    ], async (req, res) => {
        const user = await User.findByIdAndUpdate(req.params.id, {
            email: req.body.email,
            name: req.body.name
        });
        res.json(user);
    });
    """,
    # Con helmet y rate limiting
    """
    import helmet from 'helmet';
    import rateLimit from 'express-rate-limit';
    
    app.use(helmet());
    const limiter = rateLimit({
        windowMs: 15 * 60 * 1000,
        max: 100
    });
    app.use(limiter);
    
    router.post('/api/data', async (req, res) => {
        const data = await processData(req.body);
        res.json(data);
    });
    """,
]

# Crear dataset
dataset = []

for code in vulnerable_code_samples:
    dataset.append({'code': code, 'label': 'VULNERABLE'})

for code in secure_code_samples:
    dataset.append({'code': code, 'label': 'SEGURO'})

df = pd.DataFrame(dataset)

print(f"✅ Dataset creado con {len(df)} ejemplos")
print(f"   - Vulnerables: {len(df[df['label'] == 'VULNERABLE'])}")
print(f"   - Seguros: {len(df[df['label'] == 'SEGURO'])}")
print("\nDistribución de clases:")
print(df['label'].value_counts())

In [ ]:
# Importar el extractor de características
sys.path.append(os.path.dirname(os.path.abspath('__file__')))

# Importar manualmente las funciones de extracción
def extract_features_simple(code: str) -> Dict[str, float]:
    """Extracción simplificada de características"""
    features = {}
    
    # Conteo de tokens básicos
    features['num_lines'] = len(code.split('\n'))
    features['code_length'] = len(code)
    features['num_imports'] = len(re.findall(r'\b(?:import|require)\b', code))
    
    # Llamadas peligrosas
    features['exec_calls'] = len(re.findall(r'\bexec(?:Sync)?\s*\(', code))
    features['spawn_calls'] = len(re.findall(r'\bspawn(?:Sync)?\s*\(', code))
    features['eval_calls'] = len(re.findall(r'\beval\s*\(', code))
    features['function_constructor'] = len(re.findall(r'\bnew\s+Function\s*\(', code))
    
    # Operaciones de sistema de archivos
    features['fs_read'] = len(re.findall(r'\breadFile(?:Sync)?\s*\(', code))
    features['fs_write'] = len(re.findall(r'\bwriteFile(?:Sync)?\s*\(', code))
    features['fs_unlink'] = len(re.findall(r'\bunlink(?:Sync)?\s*\(', code))
    
    # Operaciones peligrosas de BD
    features['db_where'] = len(re.findall(r'\$where', code))
    features['db_update_many'] = len(re.findall(r'\bupdateMany\s*\(', code))
    features['db_delete_many'] = len(re.findall(r'\bdeleteMany\s*\(', code))
    
    # Uso de entrada de usuario
    features['req_body'] = len(re.findall(r'req\.body', code))
    features['req_query'] = len(re.findall(r'req\.query', code))
    features['req_params'] = len(re.findall(r'req\.params', code))
    
    # Sanitización y validación
    features['has_validator'] = 1 if re.search(r'express-validator|validationResult', code) else 0
    features['has_sanitize'] = 1 if re.search(r'sanitize|xss-clean|mongo-sanitize', code) else 0
    features['validation_calls'] = len(re.findall(r'\b(?:body|query|param|check)\s*\(', code))
    features['trim_calls'] = len(re.findall(r'\.trim\s*\(', code))
    features['escape_calls'] = len(re.findall(r'\.escape\s*\(', code))
    
    # Seguridad
    features['has_helmet'] = 1 if re.search(r'helmet', code) else 0
    features['has_rate_limit'] = 1 if re.search(r'rateLimit', code) else 0
    features['has_bcrypt'] = 1 if re.search(r'bcrypt', code) else 0
    features['has_jwt'] = 1 if re.search(r'\bjwt\b', code) else 0
    features['has_auth_middleware'] = 1 if re.search(r'authenticate|authorize', code) else 0
    
    # Antipatrones
    features['has_hardcoded'] = 1 if re.search(r'hardcoded|TODO.*security', code, re.IGNORECASE) else 0
    
    # Complejidad aproximada
    features['num_functions'] = len(re.findall(r'\bfunction\b|=>\s*{|async\s+\(', code))
    features['num_conditionals'] = len(re.findall(r'\bif\s*\(', code))
    
    return features

# Extraer características de todos los ejemplos
print("Extrayendo características del dataset...")
features_list = []
labels = []

for idx, row in df.iterrows():
    features = extract_features_simple(row['code'])
    features_list.append(features)
    labels.append(1 if row['label'] == 'VULNERABLE' else 0)

# Convertir a DataFrame
features_df = pd.DataFrame(features_list)
features_df['label'] = labels

print(f"✅ Características extraídas: {features_df.shape[1] - 1} features")
print(f"\nPrimeras 5 filas:")
print(features_df.head())

In [ ]:
# Análisis de correlaciones
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Distribución de labels
features_df['label'].value_counts().plot(kind='bar', ax=axes[0, 0], color=['green', 'red'])
axes[0, 0].set_title('Distribución de Clases')
axes[0, 0].set_xlabel('Clase (0=SEGURO, 1=VULNERABLE)')
axes[0, 0].set_ylabel('Cantidad')

# 2. Llamadas peligrosas por clase
dangerous_features = ['exec_calls', 'eval_calls', 'fs_read', 'db_where']
features_df.groupby('label')[dangerous_features].sum().T.plot(kind='bar', ax=axes[0, 1])
axes[0, 1].set_title('Llamadas Peligrosas por Clase')
axes[0, 1].set_xlabel('Tipo de Llamada')
axes[0, 1].set_ylabel('Cantidad')
axes[0, 1].legend(['SEGURO', 'VULNERABLE'])

# 3. Sanitización por clase
sanitization_features = ['has_validator', 'has_sanitize', 'validation_calls']
features_df.groupby('label')[sanitization_features].sum().T.plot(kind='bar', ax=axes[1, 0])
axes[1, 0].set_title('Sanitización por Clase')
axes[1, 0].set_xlabel('Tipo de Sanitización')
axes[1, 0].set_ylabel('Cantidad')
axes[1, 0].legend(['SEGURO', 'VULNERABLE'])

# 4. Entrada de usuario por clase
user_input_features = ['req_body', 'req_query', 'req_params']
features_df.groupby('label')[user_input_features].sum().T.plot(kind='bar', ax=axes[1, 1])
axes[1, 1].set_title('Uso de Entrada de Usuario por Clase')
axes[1, 1].set_xlabel('Fuente de Entrada')
axes[1, 1].set_ylabel('Cantidad')
axes[1, 1].legend(['SEGURO', 'VULNERABLE'])

plt.tight_layout()
plt.show()

print("\n📊 Estadísticas por clase:")
print(features_df.groupby('label').describe().T)

In [ ]:
# Separar características y etiquetas
X = features_df.drop('label', axis=1)
y = features_df['label']

# División train/test (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Normalización de características
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Datos preparados:")
print(f"   - Entrenamiento: {X_train.shape[0]} ejemplos")
print(f"   - Prueba: {X_test.shape[0]} ejemplos")
print(f"   - Características: {X_train.shape[1]}")
print(f"\nDistribución en entrenamiento:")
print(y_train.value_counts())
print(f"\nDistribución en prueba:")
print(y_test.value_counts())

In [ ]:
# Definir modelos
models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        class_weight='balanced'
    ),
    'XGBoost': XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        eval_metric='logloss'
    ),
    'SVM': SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        probability=True,
        random_state=42,
        class_weight='balanced'
    ),
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced'
    )
}

# Entrenar todos los modelos
trained_models = {}
results = {}

print("🚀 Entrenando modelos...")
print("="*60)

for name, model in models.items():
    print(f"\n📊 Entrenando {name}...")
    
    # Entrenar
    model.fit(X_train_scaled, y_train)
    
    # Predecir
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else None
    
    # Métricas
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    # Guardar resultados
    trained_models[name] = model
    results[name] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    
    print(f"   Accuracy:  {accuracy:.4f}")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall:    {recall:.4f}")
    print(f"   F1-Score:  {f1:.4f}")

print("\n" + "="*60)
print("✅ Todos los modelos entrenados correctamente")

In [ ]:
# Validación cruzada estratificada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

print("🔄 Realizando validación cruzada (k=5)...")
print("="*60)

for name, model in models.items():
    print(f"\n📊 {name}:")
    
    # Validación cruzada
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='f1')
    
    cv_results[name] = {
        'mean_f1': scores.mean(),
        'std_f1': scores.std(),
        'scores': scores
    }
    
    print(f"   F1-Score medio: {scores.mean():.4f} (+/- {scores.std():.4f})")
    print(f"   Scores individuales: {[f'{s:.4f}' for s in scores]}")

print("\n" + "="*60)
print("✅ Validación cruzada completada")

# Crear DataFrame con resultados
cv_df = pd.DataFrame({
    'Model': list(cv_results.keys()),
    'Mean F1': [v['mean_f1'] for v in cv_results.values()],
    'Std F1': [v['std_f1'] for v in cv_results.values()]
}).sort_values('Mean F1', ascending=False)

print("\n📊 Resultados de Validación Cruzada:")
print(cv_df.to_string(index=False))

In [ ]:
# Crear DataFrame de comparación
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [v['accuracy'] for v in results.values()],
    'Precision': [v['precision'] for v in results.values()],
    'Recall': [v['recall'] for v in results.values()],
    'F1-Score': [v['f1_score'] for v in results.values()]
}).sort_values('F1-Score', ascending=False)

print("📊 Comparación de Modelos en Test Set:")
print(comparison_df.to_string(index=False))

# Gráficos de comparación
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Comparación de métricas
comparison_df.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1-Score']].plot(
    kind='bar', ax=axes[0, 0], rot=45
)
axes[0, 0].set_title('Comparación de Métricas por Modelo', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Score')
axes[0, 0].legend(loc='lower right')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axhline(y=0.85, color='r', linestyle='--', label='Mínimo requerido (0.85)')

# 2. F1-Score con validación cruzada
cv_comparison = pd.DataFrame({
    'Model': list(cv_results.keys()),
    'CV F1-Score': [v['mean_f1'] for v in cv_results.values()],
    'Test F1-Score': [results[k]['f1_score'] for k in cv_results.keys()]
})
cv_comparison.set_index('Model').plot(kind='bar', ax=axes[0, 1], rot=45)
axes[0, 1].set_title('F1-Score: Validación Cruzada vs Test', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('F1-Score')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axhline(y=0.85, color='r', linestyle='--')

# 3. Matriz de confusión del mejor modelo
best_model_name = comparison_df.iloc[0]['Model']
best_predictions = results[best_model_name]['predictions']
cm = confusion_matrix(y_test, best_predictions)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0],
            xticklabels=['SEGURO', 'VULNERABLE'],
            yticklabels=['SEGURO', 'VULNERABLE'])
axes[1, 0].set_title(f'Matriz de Confusión - {best_model_name}', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Real')
axes[1, 0].set_xlabel('Predicción')

# 4. Curva ROC del mejor modelo
if results[best_model_name]['probabilities'] is not None:
    fpr, tpr, _ = roc_curve(y_test, results[best_model_name]['probabilities'])
    auc_score = roc_auc_score(y_test, results[best_model_name]['probabilities'])
    axes[1, 1].plot(fpr, tpr, label=f'{best_model_name} (AUC = {auc_score:.3f})', linewidth=2)
    axes[1, 1].plot([0, 1], [0, 1], 'k--', label='Random')
    axes[1, 1].set_title('Curva ROC - Mejor Modelo', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('False Positive Rate')
    axes[1, 1].set_ylabel('True Positive Rate')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n🏆 MEJOR MODELO: {best_model_name}")
print(f"   F1-Score en Test: {results[best_model_name]['f1_score']:.4f}")

In [ ]:
# Seleccionar el mejor modelo
best_model_name = comparison_df.iloc[0]['Model']
best_model = trained_models[best_model_name]

# Exportar modelo
model_path = 'model.joblib'
scaler_path = 'scaler.joblib'
feature_names_path = 'feature_names.joblib'

joblib.dump(best_model, model_path)
joblib.dump(scaler, scaler_path)
joblib.dump(list(X.columns), feature_names_path)

print(f"✅ Modelo exportado correctamente:")
print(f"   📁 Modelo: {model_path}")
print(f"   📁 Scaler: {scaler_path}")
print(f"   📁 Features: {feature_names_path}")
print(f"\n🏆 Mejor modelo: {best_model_name}")
print(f"   F1-Score: {results[best_model_name]['f1_score']:.4f}")
print(f"   Accuracy: {results[best_model_name]['accuracy']:.4f}")

# Guardar metadatos
metadata = {
    'model_name': best_model_name,
    'f1_score': float(results[best_model_name]['f1_score']),
    'accuracy': float(results[best_model_name]['accuracy']),
    'precision': float(results[best_model_name]['precision']),
    'recall': float(results[best_model_name]['recall']),
    'num_features': int(X.shape[1]),
    'feature_names': list(X.columns),
    'training_date': pd.Timestamp.now().isoformat()
}

with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"   📁 Metadata: model_metadata.json")

In [ ]:
# Cargar el modelo exportado
loaded_model = joblib.load(model_path)
loaded_scaler = joblib.load(scaler_path)
loaded_features = joblib.load(feature_names_path)

print("✅ Modelo cargado correctamente\n")

# Código de ejemplo vulnerable
test_vulnerable_code = """
router.post('/search', async (req, res) => {
    const { username } = req.body;
    const user = await User.findOne({ $where: `this.username == '${username}'` });
    res.json(user);
});
"""

# Código de ejemplo seguro
test_secure_code = """
import { body, validationResult } from 'express-validator';
import mongoSanitize from 'express-mongo-sanitize';

router.post('/search', [
    body('username').isAlphanumeric().trim().escape()
], async (req, res) => {
    const errors = validationResult(req);
    if (!errors.isEmpty()) return res.status(400).json({ errors: errors.array() });
    
    const user = await User.findOne({ username: req.body.username });
    res.json(user);
});
"""

def predict_vulnerability(code: str, model, scaler, feature_names):
    """Predice si el código es vulnerable"""
    # Extraer características
    features = extract_features_simple(code)
    
    # Convertir a DataFrame
    features_df = pd.DataFrame([features])
    
    # Asegurar que tenemos todas las columnas
    for col in feature_names:
        if col not in features_df.columns:
            features_df[col] = 0
    
    features_df = features_df[feature_names]
    
    # Escalar
    features_scaled = scaler.transform(features_df)
    
    # Predecir
    prediction = model.predict(features_scaled)[0]
    probability = model.predict_proba(features_scaled)[0]
    
    result = "VULNERABLE" if prediction == 1 else "SEGURO"
    confidence = probability[prediction] * 100
    
    # Detectar tipo de vulnerabilidad si es vulnerable
    vulnerability_type = "N/A"
    if prediction == 1:
        if features['db_where'] > 0:
            vulnerability_type = "NoSQL Injection"
        elif features['exec_calls'] > 0 or features['spawn_calls'] > 0:
            vulnerability_type = "Command Injection"
        elif features['eval_calls'] > 0:
            vulnerability_type = "Code Injection"
        elif features['fs_read'] > 0 or features['fs_write'] > 0:
            vulnerability_type = "Path Traversal"
        elif features['req_body'] > 0 and features['validation_calls'] == 0:
            vulnerability_type = "Validación Insuficiente"
    
    return {
        'result': result,
        'confidence': confidence,
        'vulnerability_type': vulnerability_type,
        'probabilities': {
            'SEGURO': probability[0] * 100,
            'VULNERABLE': probability[1] * 100
        }
    }

# Probar con código vulnerable
print("="*70)
print("🔴 TEST 1: Código VULNERABLE (NoSQL Injection)")
print("="*70)
result1 = predict_vulnerability(test_vulnerable_code, loaded_model, loaded_scaler, loaded_features)
print(f"Resultado: {result1['result']}")
print(f"Confianza: {result1['confidence']:.2f}%")
print(f"Tipo de vulnerabilidad: {result1['vulnerability_type']}")
print(f"Probabilidades: SEGURO={result1['probabilities']['SEGURO']:.2f}%, VULNERABLE={result1['probabilities']['VULNERABLE']:.2f}%")

print("\n" + "="*70)
print("🟢 TEST 2: Código SEGURO (con validación)")
print("="*70)
result2 = predict_vulnerability(test_secure_code, loaded_model, loaded_scaler, loaded_features)
print(f"Resultado: {result2['result']}")
print(f"Confianza: {result2['confidence']:.2f}%")
print(f"Tipo de vulnerabilidad: {result2['vulnerability_type']}")
print(f"Probabilidades: SEGURO={result2['probabilities']['SEGURO']:.2f}%, VULNERABLE={result2['probabilities']['VULNERABLE']:.2f}%")

print("\n" + "="*70)
print("✅ Pruebas completadas exitosamente")
print("="*70)

## 10. Prueba del Modelo con Ejemplos

Probamos el modelo exportado con código de ejemplo para verificar su funcionamiento.

## 9. Exportación del Mejor Modelo

Exportamos el mejor modelo junto con el scaler para su uso en producción.

## 8. Visualización de Resultados y Comparación

Comparamos todos los modelos visualmente para seleccionar el mejor.

## 7. Validación Cruzada

Realizamos validación cruzada estratificada con k=5 para evaluar la robustez de los modelos.

## 6. Entrenamiento de Modelos Clasificadores

Entrenaremos cuatro modelos diferentes y compararemos su rendimiento:
- Random Forest
- XGBoost
- SVM (Support Vector Machine)
- Logistic Regression

## 5. Preparación de Datos para Entrenamiento

Dividimos el dataset en conjuntos de entrenamiento y prueba, y normalizamos las características.

## 4. Análisis Exploratorio de Características

Visualizamos la distribución de las características más relevantes.

## 3. Importar Extractor de Características

Utilizamos el módulo `feature_extraction_ts.py` que ya creamos para extraer características del código.

## 2. Creación del Dataset Sintético

Como no tenemos acceso directo a datasets públicos, crearemos un dataset sintético con ejemplos de código vulnerable y seguro basados en patrones reales.

## 1. Instalación de Dependencias

Instalamos las bibliotecas necesarias para el análisis y entrenamiento.